[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/18_capstone/18_proyecto_capstone_cinematch.ipynb)

# Proyecto 18 · Capstone: CineMatch end-to-end

**Nivel:** 🔴 Experto · **Duración:** 15–25 h (planifícalo como un sprint de 1–2 semanas) · **GPU:** T4 recomendada
para el two-tower con `SCALE="full"` (en CPU funciona con `FAST_DEV_RUN=True`) · **Unidades Colab:** 3–10

## 🏢 Contexto de negocio

Eres el/la **tech lead de recomendación de CineMatch**. El comité de dirección aprobará el lanzamiento de la nueva
home si presentas un sistema **completo, medido y operable**: retrieval multi-fuente con un two-tower propio,
ranker, re-ranking de diversidad, una API con SLO, trazabilidad en MLflow, un informe de drift con decisión de
reentreno y —como extra— un asistente conversacional. Tu entregable final es este notebook ejecutado + un
*design doc* de una página.

## 📦 Dataset

**MovieLens-1M** (`SCALE="full"`, recomendado para la entrega: 1 000 209 valoraciones, 6 040 usuarios, 3 706 películas)
o **MovieLens-100K** (`SCALE="small"`, para iterar). GroupLens: https://grouplens.org/datasets/movielens/ ·
Fallback sintético si no hay red.

## ✅ Rúbrica exigente (100 puntos + 10 de bonus)

Los umbrales son **relativos a baselines medidos en el mismo split temporal**, para que sean justos con cualquier escala.

| # | Entregable | Criterio objetivo | Puntos |
|---|---|---|---|
| E1 | Datos | split temporal global; esquema Pandera válido; test de no-leakage (todas las features usan datos < corte) | 10 |
| E2 | Retrieval | two-tower PyTorch con in-batch negatives + **corrección logQ**; Recall@100 ≥ **0,9 ×** PureSVD; recall de la **unión** con two-tower > unión sin él | 15 |
| E3 | Ranking | LambdaMART: NDCG@10 ≥ **1,10 ×** la mejor fuente sola; **ablación** de 3 grupos de features | 15 |
| E4 | Re-ranking | MMR con λ elegido: ILD@10 **+10 %** relativo con pérdida de NDCG@10 **≤ 3 %** | 10 |
| E5 | Serving | FastAPI con `/health`, `/recommend`, `/event`, `/metrics`; fallback; **p99 ≤ 100 ms** con 8 concurrentes; **paridad** offline/serving ≥ 99,9 % | 15 |
| E6 | MLOps | MLflow: runs de retrieval y ranking con métricas, modelo registrado con alias `champion`; informe de drift (OOV + Evidently o PSI) y **decisión de reentreno justificada** con el NDCG «stale vs fresh» | 15 |
| E7 | Informe | *scorecard* automático (abajo) + *design doc* (≤ 1 página: métricas, arquitectura, trade-offs, riesgos, siguientes pasos) | 10 |
| ⭐ | Bonus | agente conversacional con ≥ 3 herramientas sobre el motor (LLM con tool use) y fallback sin LLM | +10 |

**Cómo trabajar:** completa los `TODO` de cada etapa (las celdas de chequeo capturan `NotImplementedError`), y
consulta la solución de referencia solo al final. Si una etapa te bloquea, puedes usar la función de la solución y
seguir: el capstone premia el sistema completo.

In [ ]:
!pip install -q faiss-cpu lightgbm fastapi uvicorn fakeredis requests pyarrow mlflow pandera evidently

In [ ]:
import os, io, zipfile, urllib.request, time, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# LightGBM con 1 hilo: faiss-cpu trae su propia libgomp y dos runtimes OpenMP en el mismo proceso pueden bloquearse
# o ir muy lentos con muchos hilos. Con nuestros tamaños, 1–2 hilos bastan (sube LGB_THREADS si entrenas a escala).
LGB_THREADS = int(os.environ.get("LGB_THREADS", "1"))
DATA_DIR, ART_DIR = "data", "artifacts"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(ART_DIR, exist_ok=True)
ML100K_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
GENRES = ["unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
          "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
          "Romance", "Sci-Fi", "Thriller", "War", "Western"]


def synthetic_movielens(n_users=943, n_items=1682, n_ratings=100_000, seed=42):
    """Fallback sin red: imita el esquema de ML-100K (gustos por género + popularidad Zipf)."""
    rng = np.random.default_rng(seed)
    item_genres = (rng.random((n_items, len(GENRES))) < 0.12).astype(int)
    item_genres[np.arange(n_items), rng.integers(1, len(GENRES), n_items)] = 1
    user_taste = rng.dirichlet(np.ones(len(GENRES)) * 0.3, n_users)
    pop = 1.0 / np.arange(1, n_items + 1) ** 0.9
    pop = pop[rng.permutation(n_items)]
    rows, t0 = [], 874_724_710  # sept-1997, como ML-100K
    per_user = rng.multinomial(n_ratings, rng.dirichlet(np.ones(n_users)))
    for u in range(n_users):
        k = max(per_user[u], 20)
        p = pop * (item_genres @ user_taste[u] + 0.02)
        p /= p.sum()
        items = rng.choice(n_items, size=min(k, n_items // 2), replace=False, p=p)
        aff = item_genres[items] @ user_taste[u]
        r = np.clip(np.round(2.5 + 6 * aff + rng.normal(0, 0.8, len(items))), 1, 5)
        ts = t0 + np.sort(rng.integers(0, 215 * 86400, len(items)))
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": items + 1, "rating": r.astype(int), "ts": ts}))
    ratings = pd.concat(rows, ignore_index=True)
    items_df = pd.DataFrame(item_genres, columns=GENRES)
    items_df.insert(0, "title", [f"Película sintética {i + 1}" for i in range(n_items)])
    items_df.insert(0, "item_id", np.arange(1, n_items + 1))
    return ratings, items_df, "Sintético (esquema ML-100K)"


def load_ml100k():
    try:
        path = os.path.join(DATA_DIR, "ml-100k.zip")
        if not os.path.exists(path):
            urllib.request.urlretrieve(ML100K_URL, path)
        with zipfile.ZipFile(path) as z:
            ratings = pd.read_csv(z.open("ml-100k/u.data"), sep="\t",
                                  names=["user_id", "item_id", "rating", "ts"])
            raw = pd.read_csv(z.open("ml-100k/u.item"), sep="|", encoding="latin-1", header=None)
        items_df = raw[[0, 1] + list(range(5, 24))].copy()
        items_df.columns = ["item_id", "title"] + GENRES
        return ratings, items_df, "MovieLens-100K"
    except Exception as e:  # sin red / URL caída → nunca bloqueamos el notebook
        print("⚠️ No se pudo descargar ML-100K, uso datos sintéticos:", repr(e)[:120])
        return synthetic_movielens()


ratings, items_df, DATASET = load_ml100k()
ratings = ratings.sort_values("ts").reset_index(drop=True)
print(DATASET, ratings.shape, "usuarios:", ratings.user_id.nunique(), "ítems:", ratings.item_id.nunique())

In [ ]:
# SCALE = "small"     # ⇦ pon "full" (ML-1M) para la entrega → MovieLens-100K (minutos en CPU) · "full" → MovieLens-1M (más realista; sigue cabiendo en CPU)
SCALE = "small"     # ⇦ pon "full" (ML-1M) para la entrega
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"


def load_ml1m():
    path = os.path.join(DATA_DIR, "ml-1m.zip")
    if not os.path.exists(path):
        urllib.request.urlretrieve(ML1M_URL, path)
    with zipfile.ZipFile(path) as z:
        r = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python", names=["user_id", "item_id", "rating", "ts"])
        mv = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python", names=["item_id", "title", "genres"],
                         encoding="latin-1")
    g = mv.genres.str.replace("Children's", "Children").str.get_dummies("|")
    items = pd.concat([mv[["item_id", "title"]], g.reindex(columns=GENRES, fill_value=0)], axis=1)
    return r, items, "MovieLens-1M"


if SCALE == "full":
    try:
        ratings, items_df, DATASET = load_ml1m()
        ratings = ratings.sort_values("ts").reset_index(drop=True)
    except Exception as e:
        print("⚠️ ML-1M no disponible, sigo con", DATASET, repr(e)[:100])
print(DATASET, ratings.shape)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True          # False para la entrega (más épocas y usuarios de evaluación)
N_EVAL_USERS = 400 if FAST_DEV_RUN else 3000
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN)

In [ ]:
import faiss, lightgbm as lgb
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

N_USERS, N_ITEMS = int(ratings.user_id.max()) + 1, int(items_df.item_id.max()) + 1
GENRE_MAT = np.zeros((N_ITEMS, len(GENRES)), dtype=np.float32)
GENRE_MAT[items_df.item_id.values] = items_df[GENRES].values
GNORM = np.linalg.norm(GENRE_MAT, axis=1) + 1e-8
TITLES = dict(zip(items_df.item_id, items_df.title))

# Split temporal GLOBAL (módulo 01): 70 % entrenar fuentes · 10 % etiquetas del ranker · 20 % test
t_a, t_b = ratings.ts.quantile([0.7, 0.8]).values
train_a, train_b, test = ratings[ratings.ts < t_a], ratings[(ratings.ts >= t_a) & (ratings.ts < t_b)], ratings[ratings.ts >= t_b]
hist_a = train_a.groupby("user_id").item_id.apply(list).to_dict()
hist_b = ratings[ratings.ts < t_b].groupby("user_id").item_id.apply(list).to_dict()
lab_b = train_b.groupby("user_id").item_id.apply(set).to_dict()
test_pos = test.groupby("user_id").item_id.apply(set).to_dict()


def binary_matrix(df):
    return csr_matrix((np.ones(len(df), np.float32), (df.user_id, df.item_id)), shape=(N_USERS, N_ITEMS))


class Sources:
    """Generación de candidatos multi-fuente (módulos 04, 05, 08): embeddings+FAISS, co-ocurrencia y popularidad reciente."""

    def __init__(self, df, dim=64, item_emb=None):
        X = binary_matrix(df)
        if item_emb is None:                                    # PureSVD (Cremonesi 2010) si no nos dan two-tower
            _, s, vt = svds(X, k=dim, random_state=seed)
            item_emb = (vt.T * np.sqrt(s)).astype(np.float32)
        self.emb = item_emb / (np.linalg.norm(item_emb, axis=1, keepdims=True) + 1e-8)
        self.index = faiss.IndexHNSWFlat(self.emb.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
        self.index.hnsw.efSearch = 128
        self.index.add(self.emb)
        C = (X.T @ X).toarray().astype(np.float32)             # item-kNN coseno (Linden et al. 2003)
        np.fill_diagonal(C, 0); n = np.sqrt(np.asarray(X.sum(0)).ravel()) + 1e-8
        self.cooc = C / n[:, None] / n[None]
        recent = df[df.ts >= df.ts.max() - 30 * 86400]
        self.pop = recent.item_id.value_counts().index.to_numpy()

    def ann(self, hist, k):
        v = self.emb[hist[-50:]].mean(0, keepdims=True)
        v /= np.linalg.norm(v) + 1e-8
        s, i = self.index.search(v.astype(np.float32), k + len(hist))
        return i[0], s[0]

    def candidates(self, hist, k=100):
        """Devuelve {item: dict de features de fuente}. Unión de las tres fuentes (deduplicada)."""
        seen, out = set(hist), {}
        ids, sc = self.ann(hist, k)
        for r, (i, s) in enumerate([(i, s) for i, s in zip(ids, sc) if i not in seen and i > 0][:k]):
            out.setdefault(int(i), {})["ann_score"], out[int(i)]["ann_rank"] = float(s), r
        cs = self.cooc[hist[-50:]].sum(0); cs[list(seen)] = -1; cs[0] = -1
        top = np.argpartition(-cs, k)[:k]; top = top[np.argsort(-cs[top])]
        for r, i in enumerate(top):
            out.setdefault(int(i), {})["cooc_score"], out[int(i)]["cooc_rank"] = float(cs[i]), r
        for r, i in enumerate([i for i in self.pop if i not in seen][: k // 2]):
            out.setdefault(int(i), {})["pop_rank"] = r
        return out


SRC_FEATS = ["ann_score", "ann_rank", "cooc_score", "cooc_rank", "pop_rank"]
SRC_DEFAULT = {"ann_score": -1.0, "ann_rank": 999, "cooc_score": 0.0, "cooc_rank": 999, "pop_rank": 999}

In [ ]:
def item_stats_asof(t_cut):
    d = ratings[ratings.ts < t_cut]
    g = d.groupby("item_id")
    st = pd.DataFrame({"pop_cum": g.size(), "rating_mean": g.rating.mean(), "first_ts": g.ts.min(),
                       "pop_30d": d[d.ts >= t_cut - 30 * 86400].item_id.value_counts()})
    st = st.reindex(range(N_ITEMS))
    st["item_age_days"] = ((t_cut - st.first_ts) / 86400).fillna(-1)
    return st.fillna({"pop_cum": 0, "rating_mean": 3.0, "pop_30d": 0}).drop(columns="first_ts")


def user_stats_asof(t_cut):
    d = ratings[ratings.ts < t_cut]; g = d.groupby("user_id")
    prof = pd.DataFrame({"user_n": g.size().astype(float), "user_mean": g.rating.mean(),
                         "user_days_since_last": (t_cut - g.ts.max()) / 86400})
    gen = pd.DataFrame(GENRE_MAT[d.item_id.values], index=d.user_id.values).groupby(level=0).mean()
    gen = gen.div(np.linalg.norm(gen.values, axis=1) + 1e-8, axis=0)
    return prof, gen


RANK_FEATS = SRC_FEATS + ["n_sources", "pop_cum", "pop_30d", "rating_mean", "item_age_days",
                          "genre_affinity", "user_n", "user_mean", "user_days_since_last"]


def build_frame(src, hist, users, t_cut, k=100, labels=None):
    """Candidatos + features POINT-IN-TIME (todo calculado con datos < t_cut)."""
    ist = item_stats_asof(t_cut); prof, gen = user_stats_asof(t_cut)
    rows = []
    for u in users:
        if u not in hist or u not in prof.index:
            continue
        cands = src.candidates(hist[u], k)
        ids = np.fromiter(cands, dtype=int)
        f = pd.DataFrame([{**SRC_DEFAULT, **cands[i]} for i in ids])
        f["n_sources"] = [len(cands[i]) // 2 + ("pop_rank" in cands[i]) for i in ids]
        f["user_id"], f["item_id"] = u, ids
        f["genre_affinity"] = GENRE_MAT[ids] @ gen.loc[u].values / GNORM[ids]
        for c in ["user_n", "user_mean", "user_days_since_last"]:
            f[c] = prof.loc[u, c]
        rows.append(f)
    out = pd.concat(rows, ignore_index=True).join(ist, on="item_id")
    if labels is not None:
        out["label"] = [int(i in labels.get(u, ())) for u, i in zip(out.user_id, out.item_id)]
    return out


def train_lambdamart(frame, n_estimators=400):
    f = frame[frame.groupby("user_id").label.transform("max") > 0].sort_values("user_id")
    m = lgb.LGBMRanker(objective="lambdarank", n_estimators=n_estimators, learning_rate=0.05, num_leaves=63,
                       min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=seed, n_jobs=LGB_THREADS)
    m.fit(f[RANK_FEATS], f.label, group=f.groupby("user_id").size().values)
    return m


def ndcg_at_k(ranked, relevant, k=10):
    dcg = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in relevant)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(k, len(relevant))))
    return dcg / idcg if idcg else 0.0


def mmr(items, scores, emb, k=10, lam=0.7):
    """Maximal Marginal Relevance (Carbonell & Goldstein 1998; módulo 13): λ·relevancia − (1−λ)·máx similitud."""
    items = list(items)
    rel = (scores - scores.min()) / (np.ptp(scores) + 1e-9)
    chosen, cand = [], list(range(len(items)))
    E = emb[np.asarray(items)]
    while cand and len(chosen) < k:
        if chosen:
            sim = (E[cand] @ E[chosen].T).max(1)
        else:
            sim = np.zeros(len(cand))
        best = cand[int(np.argmax(lam * rel[cand] - (1 - lam) * sim))]
        chosen.append(best); cand.remove(best)
    return [items[c] for c in chosen]


def ild(items, emb):
    """Intra-list diversity: 1 − similitud coseno media entre pares."""
    E = emb[np.asarray(items)]; S = E @ E.T; n = len(items)
    return float(1 - (S.sum() - np.trace(S)) / (n * (n - 1)))

## Etapa 1 · Datos (E1)

`CORE_CELL` ya hizo el split temporal 70/10/20. Tu tarea: (a) esquema Pandera para `ratings`; (b) `assert_no_leakage(frame, t_cut)`
que compruebe que las features de ítem de un frame construido con `build_frame(..., t_cut)` coinciden con las calculadas
**solo** con datos anteriores a `t_cut` (p. ej. `pop_cum` == conteo de interacciones previas).

In [ ]:
def ratings_schema():
    # TODO: pa.DataFrameSchema con user_id/item_id ≥ 1, rating ∈ {1..5}, ts > 0 y sin duplicados (user, item)
    raise NotImplementedError


def assert_no_leakage(frame, t_cut):
    # TODO
    raise NotImplementedError


try:
    ratings_schema().validate(ratings[["user_id", "item_id", "rating", "ts"]]); print("E1 · esquema OK")
except NotImplementedError:
    print("⏳ E1 pendiente")

## Etapa 2 · Retrieval two-tower (E2)

Implementa `TwoTower`: torre de usuario = `EmbeddingBag` (media) sobre los últimos 50 ítems del historial → MLP →
normalización L2; torre de ítem = embedding de ID + géneros → MLP → L2. Entrena con **softmax in-batch** a
temperatura τ y **corrección logQ** (resta $\log q_j$ al logit del ítem $j$, con $q_j$ su frecuencia de muestreo;
Yi et al. 2019). Ejemplos: para cada usuario de `train_a`, posiciones $j$ de su secuencia temporal → (historial
previo, ítem $j$). Exporta la torre de usuario con TorchScript (la usará el servicio).

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_items, genre_mat, dim=64):
        super().__init__()
        # TODO: embeddings, EmbeddingBag(mode="mean", padding_idx=0), MLPs
        raise NotImplementedError

    def user(self, hist):            # hist: LongTensor (B, 50) con padding 0
        raise NotImplementedError

    def item(self, ids):
        raise NotImplementedError


def train_two_tower(df, epochs=3, dim=64, tau=0.05, batch=512, logq=True):
    # TODO: ejemplos (historial previo → siguiente ítem), bucle de entrenamiento, devuelve (modelo, emb_items normalizados)
    raise NotImplementedError

## Etapas 3–4 · Ranking, ablación y re-ranking (E3, E4)

Reutiliza `build_frame` + `train_lambdamart`. Añade las features del two-tower (`tt_score`, `tt_rank`). Ablación:
(A) solo fuentes, (B) + ítem, (C) + usuario/cruce. Re-ranking: barre λ ∈ {1,0; 0,9; …; 0,5} y elige el menor λ que
cumpla E4.

## Etapa 5 · Serving (E5)

El motor de serving (`cinematch_serving.Recommender`) y la API (`capstone_api.py`) se escriben en ficheros. Completa
`export_artifacts()` (todo lo que el motor necesita, as-of `t_b`) y el test de carga.

## Etapa 6 · MLOps (E6) y Etapa 7 · Informe (E7)

Registra en MLflow, genera el informe de drift y la decisión, y rellena el *scorecard*. La solución completa sigue.

---

# ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa y ejecutable de arriba a abajo.

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa
from pandera import Check


def ratings_schema():
    return pa.DataFrameSchema(
        {"user_id": pa.Column(int, Check.ge(1)), "item_id": pa.Column(int, Check.ge(1)),
         "rating": pa.Column(float, Check.isin([1.0, 2.0, 3.0, 4.0, 5.0])), "ts": pa.Column(int, Check.gt(0))},
        checks=[Check(lambda d: not d.duplicated(["user_id", "item_id"]).any(), error="duplicados (user,item)")], coerce=True)


def assert_no_leakage(frame, t_cut):
    true_pop = ratings[ratings.ts < t_cut].item_id.value_counts()
    got = frame.drop_duplicates("item_id").set_index("item_id").pop_cum
    ok = np.allclose(got.values, true_pop.reindex(got.index).fillna(0).values)
    assert ok, "¡pop_cum usa datos posteriores al corte!"
    return ok


E = {}
ratings_schema().validate(ratings[["user_id", "item_id", "rating", "ts"]])
E["E1"] = True
print("E1 · esquema válido · split:", len(train_a), len(train_b), len(test))

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_items, genre_mat, dim=64):
        super().__init__()
        self.register_buffer("genre", torch.tensor(genre_mat))
        self.item_emb = nn.Embedding(n_items, dim, padding_idx=0)
        self.hist_emb = nn.EmbeddingBag(n_items, dim, mode="mean", padding_idx=0)
        g = genre_mat.shape[1]
        self.user_mlp = nn.Sequential(nn.Linear(dim, 128), nn.ReLU(), nn.Linear(128, dim))
        self.item_mlp = nn.Sequential(nn.Linear(dim + g, 128), nn.ReLU(), nn.Linear(128, dim))
        nn.init.normal_(self.item_emb.weight, std=0.05); nn.init.normal_(self.hist_emb.weight, std=0.05)

    def user(self, hist):
        return F.normalize(self.user_mlp(self.hist_emb(hist)), dim=-1)

    def item(self, ids):
        return F.normalize(self.item_mlp(torch.cat([self.item_emb(ids), self.genre[ids]], -1)), dim=-1)


class UserTower(nn.Module):
    """Solo la torre de usuario, para exportar con TorchScript al servicio."""
    def __init__(self, tt):
        super().__init__(); self.hist_emb, self.user_mlp = tt.hist_emb, tt.user_mlp

    def forward(self, hist):
        return F.normalize(self.user_mlp(self.hist_emb(hist)), dim=-1)


def make_examples(df, max_per_user=40, L=50):
    r_ = np.random.default_rng(seed); H, T = [], []
    for _, g in df.sort_values("ts").groupby("user_id"):
        seq = g.item_id.to_numpy()
        if len(seq) < 3:
            continue
        pos = r_.choice(np.arange(1, len(seq)), min(max_per_user, len(seq) - 1), replace=False)
        for j in pos:
            h = seq[max(0, j - L):j]; H.append(np.pad(h, (0, L - len(h)))); T.append(seq[j])
    return torch.tensor(np.array(H)), torch.tensor(np.array(T))

In [ ]:
def train_two_tower(df, epochs=None, dim=64, tau=0.05, batch=512, logq=True, lr=3e-3):
    epochs = epochs or (4 if FAST_DEV_RUN else 15)
    H, T = make_examples(df)
    q = np.bincount(T.numpy(), minlength=N_ITEMS) / len(T)                  # frecuencia de muestreo del ítem
    logq_t = torch.tensor(np.log(q + 1e-10), dtype=torch.float32, device=device)
    model = TwoTower(N_ITEMS, GENRE_MAT, dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr); losses = []
    for ep in range(epochs):
        perm = torch.randperm(len(T)); tot = 0.0
        for b in range(0, len(T), batch):
            idx = perm[b:b + batch]; h, t = H[idx].to(device), T[idx].to(device)
            logits = model.user(h) @ model.item(t).T / tau
            if logq:
                logits = logits - logq_t[t][None, :]                        # corrección logQ (Yi et al. 2019)
            same = (t[:, None] == t[None, :]) & ~torch.eye(len(t), dtype=torch.bool, device=device)
            logits = logits.masked_fill(same, -1e9)                         # "accidental hits" = no son negativos
            loss = F.cross_entropy(logits, torch.arange(len(t), device=device))
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
        losses.append(tot / len(T))
    model.eval()
    with torch.no_grad():
        item_emb = model.item(torch.arange(N_ITEMS, device=device)).cpu().numpy()
    return model, item_emb, losses


t0 = time.time()
TT, TT_EMB, tt_losses = train_two_tower(train_a)
print(f"Two-tower entrenado en {time.time() - t0:.0f}s · loss final {tt_losses[-1]:.3f}")
plt.figure(figsize=(6, 3)); plt.plot(tt_losses, "o-"); plt.title("Two-tower: loss in-batch softmax (logQ)"); plt.xlabel("época"); plt.show()

In [ ]:
def make_user_fn(model):
    def fn(hist):
        h = hist[-50:]; x = torch.tensor([h + [0] * (50 - len(h))], device=device)
        with torch.no_grad():
            return model.user(x).cpu().numpy()
    return fn


class Sources2(Sources):
    """Sources + fuente two-tower (features tt_score / tt_rank)."""
    def __init__(self, df, tt_model=None, tt_emb=None):
        super().__init__(df)
        self.tt_fn = make_user_fn(tt_model) if tt_model is not None else None
        if tt_emb is not None:
            self.tt_emb = tt_emb.astype(np.float32)
            self.tt_index = faiss.IndexHNSWFlat(tt_emb.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
            self.tt_index.hnsw.efSearch = 128; self.tt_index.add(self.tt_emb)

    def candidates(self, hist, k=100):
        out = super().candidates(hist, k)
        if self.tt_fn is not None:
            s, i = self.tt_index.search(self.tt_fn(hist).astype(np.float32), k + len(hist)); seen = set(hist)
            for r, (it, sc) in enumerate([(a, b) for a, b in zip(i[0], s[0]) if a not in seen and a > 0][:k]):
                out.setdefault(int(it), {})["tt_score"], out[int(it)]["tt_rank"] = float(sc), r
        return out


SRC_FEATS = SRC_FEATS + ["tt_score", "tt_rank"]
SRC_DEFAULT = {**SRC_DEFAULT, "tt_score": -1.0, "tt_rank": 999}
RANK_FEATS = SRC_FEATS + [f for f in RANK_FEATS if f not in SRC_FEATS]
SRC_A = Sources2(train_a, TT, TT_EMB)

In [ ]:
# Fuentes "frescas" para servir el test: PureSVD/co-ocurrencia re-entrenadas hasta t_b; two-tower afinado hasta t_b
TT_B, TT_EMB_B, _ = train_two_tower(ratings[ratings.ts < t_b])
SRC_B = Sources2(ratings[ratings.ts < t_b], TT_B, TT_EMB_B)
users_eval = [u for u in test_pos if u in hist_b][:N_EVAL_USERS]


def source_recalls(src, users, hist):
    out = {k: [] for k in ["PureSVD", "Two-tower", "Co-ocurrencia", "Popularidad", "Unión sin TT", "Unión"]}
    for u in users:
        c = src.candidates(hist[u], 100); tp = test_pos[u]; r = lambda s: len(s & tp) / len(tp)
        sets = {k: {i for i, f in c.items() if key in f} for k, key in
                [("PureSVD", "ann_rank"), ("Two-tower", "tt_rank"), ("Co-ocurrencia", "cooc_rank"), ("Popularidad", "pop_rank")]}
        for k, s in sets.items():
            out[k].append(r(s))
        out["Unión sin TT"].append(r(sets["PureSVD"] | sets["Co-ocurrencia"] | sets["Popularidad"])); out["Unión"].append(r(set(c)))
    return {k: float(np.mean(v)) for k, v in out.items()}


rec = source_recalls(SRC_B, users_eval, hist_b)
E["E2"] = rec["Two-tower"] >= 0.9 * rec["PureSVD"] and rec["Unión"] > rec["Unión sin TT"]
print({k: round(v, 4) for k, v in rec.items()}, "→ E2", "✅" if E["E2"] else "❌")
pd.Series(rec).plot.bar(figsize=(9, 3.5), color="#5dade2", rot=15, title="E2 · Recall@100 por fuente (test temporal)"); plt.show()

In [ ]:
users_b = [u for u in lab_b if u in hist_a]
fr_tr = build_frame(SRC_A, hist_a, users_b, t_a, labels=lab_b)
assert_no_leakage(fr_tr, t_a)
fr_te = build_frame(SRC_B, hist_b, users_eval, t_b)
groups = {"A · solo fuentes": SRC_FEATS + ["n_sources"],
          "B · + ítem": SRC_FEATS + ["n_sources", "pop_cum", "pop_30d", "rating_mean", "item_age_days"],
          "C · + usuario/cruce (todas)": RANK_FEATS}


def eval_ranker(model, feats, frame):
    frame = frame.assign(score=model.predict(frame[feats]))
    return float(np.mean([ndcg_at_k(list(g.sort_values("score", ascending=False).item_id), test_pos[u])
                          for u, g in frame.groupby("user_id")])), frame


abl = {}
for name, feats in groups.items():
    RANK_FEATS_BAK, RANK_FEATS = RANK_FEATS, feats
    m = train_lambdamart(fr_tr); RANK_FEATS = RANK_FEATS_BAK
    abl[name], _ = eval_ranker(m, feats, fr_te)
RANKER = train_lambdamart(fr_tr)
nd_rank, fr_te = eval_ranker(RANKER, RANK_FEATS, fr_te)
best_src = max(float(np.mean([ndcg_at_k(list(g.sort_values(c).item_id), test_pos[u]) for u, g in fr_te.groupby("user_id")]))
               for c in ["ann_rank", "tt_rank", "cooc_rank"])
E["E3"] = nd_rank >= 1.10 * best_src and len(abl) == 3
print(f"E3 · NDCG@10 ranker = {nd_rank:.4f} vs mejor fuente {best_src:.4f} (×{nd_rank / best_src:.2f}) →", "✅" if E["E3"] else "❌")
display(pd.Series(abl, name="NDCG@10 (ablación)").round(4).to_frame())
E["E1"] = E["E1"] and True

In [ ]:
lams, rows = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5], []
G_te = list(fr_te.groupby("user_id"))
for lam in lams:
    nd, dv = [], []
    for u, g in G_te:
        g = g.sort_values("score", ascending=False)
        rr = mmr(g.item_id.values[:50], g.score.values[:50], SRC_B.emb, k=10, lam=lam)
        nd.append(ndcg_at_k(rr, test_pos[u])); dv.append(ild(rr, SRC_B.emb))
    rows.append({"lambda": lam, "NDCG@10": np.mean(nd), "ILD@10": np.mean(dv)})
mm = pd.DataFrame(rows); base = mm.iloc[0]
ok = mm[(mm["ILD@10"] >= 1.10 * base["ILD@10"]) & (mm["NDCG@10"] >= 0.97 * base["NDCG@10"])]
LAMBDA = float(ok["lambda"].max()) if len(ok) else 0.8
E["E4"] = len(ok) > 0
display(mm.round(4)); print(f"E4 · λ elegido = {LAMBDA} →", "✅" if E["E4"] else "❌ (ninguna λ cumple: prueba otra similitud, p. ej. géneros)")
fig, ax = plt.subplots(figsize=(6.5, 3.8)); ax.plot(mm["ILD@10"], mm["NDCG@10"], "o-")
for _, r in mm.iterrows():
    ax.annotate(f"λ={r['lambda']}", (r["ILD@10"], r["NDCG@10"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("ILD@10"); ax.set_ylabel("NDCG@10"); ax.set_title("E4 · Frontera MMR"); plt.show()

In [ ]:
def export_artifacts(art=ART_DIR):
    os.makedirs(art, exist_ok=True)
    np.save(f"{art}/genre_mat.npy", GENRE_MAT); np.save(f"{art}/svd_emb.npy", SRC_B.emb)
    faiss.write_index(SRC_B.index, f"{art}/svd.faiss"); np.save(f"{art}/cooc.npy", SRC_B.cooc)
    np.save(f"{art}/pop_recent.npy", SRC_B.pop); np.save(f"{art}/tt_item_emb.npy", SRC_B.tt_emb)
    faiss.write_index(SRC_B.tt_index, f"{art}/tt.faiss")
    ut = torch.jit.trace(UserTower(TT_B).cpu().eval(), torch.zeros(1, 50, dtype=torch.long)); ut.save(f"{art}/tt_user.pt")
    TT_B.to(device)
    ist = item_stats_asof(t_b); ist.to_parquet(f"{art}/item_stats.parquet")
    prof, gen = user_stats_asof(t_b); prof.to_parquet(f"{art}/user_prof.parquet")
    gen.columns = [str(c) for c in gen.columns]; gen.to_parquet(f"{art}/user_genre.parquet")
    RANKER.booster_.save_model(f"{art}/ranker.txt")
    json.dump({str(u): [int(x) for x in h[-50:]] for u, h in hist_b.items()}, open(f"{art}/user_hist.json", "w"))
    json.dump({"rank_feats": RANK_FEATS, "src_default": SRC_DEFAULT, "lambda": LAMBDA, "model_version": "capstone-v1",
               "popular": [int(i) for i in ratings[ratings.ts < t_b].item_id.value_counts().index[:200]]},
              open(f"{art}/meta.json", "w"))
    return sorted(os.listdir(art))


print(export_artifacts())

In [ ]:
%%writefile cinematch_serving.py
"""CineMatch · motor de recomendación de serving (capstone). Lo usan la API y el notebook (paridad por construcción)."""
import json, os, time
import faiss, lightgbm as lgb, numpy as np, pandas as pd


class Recommender:
    def __init__(self, art="artifacts"):
        self.meta = json.load(open(f"{art}/meta.json"))
        self.feats, self.default = self.meta["rank_feats"], self.meta["src_default"]
        self.genre = np.load(f"{art}/genre_mat.npy"); self.gnorm = np.linalg.norm(self.genre, axis=1) + 1e-8
        self.svd = np.load(f"{art}/svd_emb.npy"); self.svd_index = faiss.read_index(f"{art}/svd.faiss")
        self.svd_index.hnsw.efSearch = 128
        self.cooc = np.load(f"{art}/cooc.npy"); self.pop = np.load(f"{art}/pop_recent.npy")
        self.tt_user = None
        if os.path.exists(f"{art}/tt_user.pt"):
            import torch
            self.torch = torch; torch.set_num_threads(1)
            self.tt_user = torch.jit.load(f"{art}/tt_user.pt").eval()
            self.tt_emb = np.load(f"{art}/tt_item_emb.npy"); self.tt_index = faiss.read_index(f"{art}/tt.faiss")
            self.tt_index.hnsw.efSearch = 128
        ist = pd.read_parquet(f"{art}/item_stats.parquet")
        self.ist = {c: ist[c].to_numpy(float) for c in ist.columns}
        prof = pd.read_parquet(f"{art}/user_prof.parquet")
        self.prof = {int(u): r for u, r in zip(prof.index, prof.to_numpy(float))}; self.prof_cols = list(prof.columns)
        ugen = pd.read_parquet(f"{art}/user_genre.parquet")
        self.ugen = {int(u): r for u, r in zip(ugen.index, ugen.to_numpy())}
        self.ranker = lgb.Booster(model_file=f"{art}/ranker.txt")
        self.popular = self.meta["popular"]

    def _ann(self, index, v, hist, k, prefix, out):
        s, i = index.search(v.astype(np.float32), k + len(hist)); seen = set(hist)
        for r, (it, sc) in enumerate([(a, b) for a, b in zip(i[0], s[0]) if a not in seen and a > 0][:k]):
            out.setdefault(int(it), {})[f"{prefix}_score"], out[int(it)][f"{prefix}_rank"] = float(sc), r

    def tt_vector(self, hist):
        h = hist[-50:]; x = self.torch.tensor([h + [0] * (50 - len(h))])
        with self.torch.no_grad():
            return self.tt_user(x).numpy()

    def candidates(self, hist, k=100):
        out, seen = {}, set(hist)
        v = self.svd[hist[-50:]].mean(0, keepdims=True); v /= np.linalg.norm(v) + 1e-8
        self._ann(self.svd_index, v, hist, k, "ann", out)
        cs = self.cooc[hist[-50:]].sum(0); cs[list(seen)] = -1; cs[0] = -1
        top = np.argpartition(-cs, k)[:k]; top = top[np.argsort(-cs[top])]
        for r, i in enumerate(top):
            out.setdefault(int(i), {})["cooc_score"], out[int(i)]["cooc_rank"] = float(cs[i]), r
        for r, i in enumerate([i for i in self.pop if i not in seen][: k // 2]):
            out.setdefault(int(i), {})["pop_rank"] = r
        if self.tt_user is not None:
            self._ann(self.tt_index, self.tt_vector(hist), hist, k, "tt", out)
        return out

    def features(self, u, cands):
        ids = np.fromiter(cands, dtype=int); cols = {}
        for f in self.default:
            cols[f] = np.array([cands[i].get(f, self.default[f]) for i in ids], dtype=float)
        cols["n_sources"] = np.array([len(cands[i]) // 2 + ("pop_rank" in cands[i]) for i in ids], dtype=float)
        cols["genre_affinity"] = self.genre[ids] @ self.ugen[u] / self.gnorm[ids]
        for j, c in enumerate(self.prof_cols):
            cols[c] = np.full(len(ids), self.prof[u][j])
        for c, arr in self.ist.items():
            cols[c] = arr[ids]
        return ids, np.column_stack([cols[f] for f in self.feats])

    def recommend(self, u, hist, k=10, lam=0.8):
        t = [time.perf_counter()]
        if not hist or u not in self.prof:
            return [i for i in self.popular if i not in set(hist)][:k], {"strategy": "popularity_fallback"}
        cands = self.candidates(hist); t.append(time.perf_counter())
        ids, X = self.features(u, cands); t.append(time.perf_counter())
        s = self.ranker.predict(X, num_threads=1); o = np.argsort(-s)[:50]; t.append(time.perf_counter())
        items = self.mmr(ids[o], s[o], k, lam); t.append(time.perf_counter())
        names = ["retrieval", "features", "ranking", "reranking"]
        return items, {"strategy": "multi_stage", **{n: 1e3 * (b - a) for n, a, b in zip(names, t, t[1:])}}

    def mmr(self, items, scores, k, lam):
        rel = (scores - scores.min()) / (np.ptp(scores) + 1e-9); E = self.svd[items]
        chosen, cand = [], list(range(len(items)))
        while cand and len(chosen) < k:
            sim = (E[cand] @ E[chosen].T).max(1) if chosen else np.zeros(len(cand))
            best = cand[int(np.argmax(lam * rel[cand] - (1 - lam) * sim))]
            chosen.append(best); cand.remove(best)
        return [int(items[c]) for c in chosen]

In [ ]:
%%writefile capstone_api.py
"""API de CineMatch (capstone): FastAPI sobre el motor `Recommender`."""
import json, os, threading, time
import fakeredis
from fastapi import FastAPI, Response
from pydantic import BaseModel
from cinematch_serving import Recommender

ART = os.environ.get("ART_DIR", "artifacts")
REC = Recommender(ART)
R = fakeredis.FakeRedis(decode_responses=True)
for u, h in json.load(open(f"{ART}/user_hist.json")).items():
    R.rpush(f"hist:{u}", *h)
TTL = int(os.environ.get("CACHE_TTL", "60"))
LOCK = threading.Lock()
M = {"requests_total": 0, "cache_hits_total": 0, "fallback_total": 0, "latency_ms_sum": 0.0}
app = FastAPI(title="CineMatch capstone", version=REC.meta["model_version"])


class Event(BaseModel):
    user_id: int
    item_id: int


@app.get("/health")
def health():
    return {"status": "ok", "model_version": REC.meta["model_version"]}


@app.get("/recommend/{user_id}")
def recommend(user_id: int, k: int = 10, use_cache: bool = True):
    t0 = time.perf_counter()
    with LOCK:
        M["requests_total"] += 1
    if use_cache and (hit := R.get(f"recs:{user_id}:{k}")):
        with LOCK:
            M["cache_hits_total"] += 1
        return {**json.loads(hit), "cache": True}
    hist = [int(x) for x in R.lrange(f"hist:{user_id}", -50, -1)]
    items, info = REC.recommend(user_id, hist, k)
    if info["strategy"] == "popularity_fallback":
        with LOCK:
            M["fallback_total"] += 1
    resp = {"user_id": user_id, "items": items, "model_version": REC.meta["model_version"], **info}
    R.setex(f"recs:{user_id}:{k}", TTL, json.dumps(resp))
    with LOCK:
        M["latency_ms_sum"] += 1e3 * (time.perf_counter() - t0)
    return {**resp, "cache": False}


@app.post("/event")
def event(e: Event):
    R.rpush(f"hist:{e.user_id}", e.item_id); R.ltrim(f"hist:{e.user_id}", -50, -1)
    for key in R.scan_iter(f"recs:{e.user_id}:*"):
        R.delete(key)
    return {"ok": True}


@app.get("/metrics")
def metrics():
    return Response("".join(f"cinematch_{k} {v}\n" for k, v in M.items()), media_type="text/plain")

In [ ]:
import importlib, cinematch_serving
importlib.reload(cinematch_serving)
ENGINE = cinematch_serving.Recommender(ART_DIR)

# Paridad offline/serving: mismas features para los mismos (usuario, ítem)
par = []
for u in users_eval[:100]:
    ids, X = ENGINE.features(u, ENGINE.candidates(hist_b[u]))
    off = fr_te[fr_te.user_id == u].set_index("item_id").loc[ids, RANK_FEATS].to_numpy(float)
    par.append(np.isclose(X, off, atol=1e-4).mean())
parity = float(np.mean(par))
nd_engine = np.mean([ndcg_at_k(ENGINE.recommend(u, hist_b[u], 10, LAMBDA)[0], test_pos[u]) for u in users_eval[:300]])
print(f"Paridad de features offline vs motor de serving: {parity:.4%} · NDCG@10 del motor (λ={LAMBDA}) = {nd_engine:.4f}")

In [ ]:
import subprocess, sys, requests
from concurrent.futures import ThreadPoolExecutor

proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "capstone_api:app", "--port", "8018", "--log-level", "warning"],
                        env={**os.environ, "ART_DIR": ART_DIR})
URL = "http://127.0.0.1:8018"
for _ in range(240):
    try:
        if requests.get(f"{URL}/health", timeout=0.5).ok:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.5)
u0 = users_eval[0]
a = requests.get(f"{URL}/recommend/{u0}").json(); b = requests.get(f"{URL}/recommend/{u0}").json()
requests.post(f"{URL}/event", json={"user_id": u0, "item_id": a["items"][0]})
c = requests.get(f"{URL}/recommend/{u0}").json(); cold = requests.get(f"{URL}/recommend/987654321").json()
api_ok = b["cache"] and not c["cache"] and cold["strategy"] == "popularity_fallback" and "cinematch_requests_total" in requests.get(f"{URL}/metrics").text


def load_test(conc, n=300):
    s = requests.Session()
    def one(u):
        t = time.perf_counter(); s.get(f"{URL}/recommend/{u}", params={"use_cache": False}); return 1e3 * (time.perf_counter() - t)
    us = np.random.default_rng(seed).choice(users_eval, n); t0 = time.perf_counter()
    with ThreadPoolExecutor(conc) as ex:
        lat = np.array(list(ex.map(one, us)))
    return {"conc": conc, "qps": n / (time.perf_counter() - t0), "p50": np.percentile(lat, 50), "p99": np.percentile(lat, 99)}


[requests.get(f"{URL}/recommend/{u}", params={"use_cache": False}) for u in users_eval[:20]]   # warm-up
lt = pd.DataFrame([load_test(c) for c in [1, 4, 8]]); display(lt.round(1))
p99_8 = float(lt[lt.conc == 8].p99.iloc[0])
E["E5"] = api_ok and p99_8 <= 100 and parity >= 0.999
print(f"E5 · API ok={api_ok} · p99@8 = {p99_8:.1f} ms · paridad = {parity:.4%} →", "✅" if E["E5"] else "❌")

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("sqlite:///mlflow_capstone.db"); mlflow.set_experiment("cinematch-capstone")
mc = MlflowClient()


class CineMatchPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import sys as _s; _s.path.insert(0, ".")
        from cinematch_serving import Recommender
        self.rec = Recommender(context.artifacts["art"])

    def predict(self, context, model_input, params=None):
        return pd.DataFrame({"items": [self.rec.recommend(int(u), list(h))[0] for u, h in zip(model_input.user_id, model_input.history)]})


with mlflow.start_run(run_name="retrieval"):
    mlflow.log_params({"tt_dim": 64, "tau": 0.05, "logq": True, "scale": SCALE}); mlflow.log_metrics({f"recall100_{k.replace(' ', '_')}": v for k, v in rec.items()})
with mlflow.start_run(run_name="ranking+serving") as run:
    mlflow.log_params({"ranker": "LGBMRanker-lambdarank", "mmr_lambda": LAMBDA, "n_feats": len(RANK_FEATS)})
    mlflow.log_metrics({"ndcg10_ranker": nd_rank, "ndcg10_engine": float(nd_engine), "p99_ms_conc8": p99_8, "parity": parity})
    kw = dict(python_model=CineMatchPyfunc(), artifacts={"art": ART_DIR}, registered_model_name="cinematch-capstone",
              code_paths=["cinematch_serving.py"])
    try:
        mlflow.pyfunc.log_model(name="model", **kw)
    except TypeError:
        mlflow.pyfunc.log_model(artifact_path="model", **kw)
v = max(int(x.version) for x in mc.search_model_versions("name='cinematch-capstone'"))
mc.set_registered_model_alias("cinematch-capstone", "champion", v)
print("Registrado cinematch-capstone v", v, "con alias champion")

In [ ]:
# Informe de drift + decisión de reentreno basada en evidencia (stale vs fresh)
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    p = np.histogram(np.clip(ref, edges[0], edges[-1]), edges)[0] / len(ref) + 1e-4
    q = np.histogram(np.clip(cur, edges[0], edges[-1]), edges)[0] / len(cur) + 1e-4
    return float(np.sum((p - q) * np.log(p / q)))


pop_a = train_a.item_id.value_counts()
drift = {"oov_test_vs_train_a": float((~test.item_id.isin(pop_a.index)).mean()),
         "psi_logpop": psi(np.log1p(train_a.item_id.map(pop_a).values), np.log1p(test.item_id.map(pop_a).fillna(0).values)),
         "psi_rating": psi(train_a.rating.values.astype(float), test.rating.values.astype(float))}
try:
    from evidently import Report
    from evidently.presets import DataDriftPreset
    mk = lambda d: pd.DataFrame({"rating": d.rating.astype(float).values, "log_pop": np.log1p(d.item_id.map(pop_a).fillna(0)).values})
    Report([DataDriftPreset()]).run(current_data=mk(test), reference_data=mk(train_a)).save_html("capstone_drift.html")
    drift["evidently_html"] = "capstone_drift.html"
except Exception as e:
    print("Evidently no disponible:", repr(e)[:100])
# ¿Cuánto perdemos sirviendo con fuentes entrenadas hasta t_a (stale) en vez de t_b (fresh)?
fr_stale = build_frame(SRC_A, hist_b, users_eval[:300], t_b)
nd_stale, _ = eval_ranker(RANKER, RANK_FEATS, fr_stale)
nd_fresh, _ = eval_ranker(RANKER, RANK_FEATS, fr_te[fr_te.user_id.isin(users_eval[:300])])
decision = "REENTRENAR (frescura vale ≥ 3 %)" if nd_fresh >= 1.03 * nd_stale else "mantener calendario actual"
drift.update({"ndcg_stale": nd_stale, "ndcg_fresh": nd_fresh, "decision": decision})
with mlflow.start_run(run_name="drift-report"):
    mlflow.log_metrics({k: v for k, v in drift.items() if isinstance(v, float)}); mlflow.set_tag("decision", decision)
E["E6"] = True
display(pd.Series(drift).to_frame("valor"))

### ⭐ Bonus · Asistente conversacional sobre el motor

El agente expone el motor como **herramientas** (módulo 12). Si hay `ANTHROPIC_API_KEY` y el paquete `anthropic`,
usa la Claude API con *tool use* (bucle manual: el modelo pide herramientas, las ejecutamos y le devolvemos los
resultados); si no, un *fallback* determinista enruta por palabras clave a las mismas herramientas — el notebook nunca
se bloquea. El modelo se configura con `CINEMATCH_LLM_MODEL`.

In [ ]:
GENRE_IDX = {g.lower(): j for j, g in enumerate(GENRES)}


def tool_recommend_for_user(user_id: int, k: int = 5):
    return [TITLES.get(i, str(i)) for i in ENGINE.recommend(int(user_id), hist_b.get(int(user_id), []), int(k), LAMBDA)[0]]


def tool_search_by_genre(genre: str, k: int = 5):
    j = GENRE_IDX.get(genre.lower())
    if j is None:
        return {"error": f"género desconocido; usa uno de {GENRES[1:]}"}
    pool = [i for i in ENGINE.popular if GENRE_MAT[i, j] > 0][:k]
    return [TITLES.get(i, str(i)) for i in pool]


def tool_similar_to(title: str, k: int = 5):
    match = [i for i, t in TITLES.items() if title.lower() in str(t).lower()]
    if not match:
        return {"error": "título no encontrado"}
    v = SRC_B.emb[match[0]][None]; _, ids = SRC_B.index.search(v.astype(np.float32), k + 1)
    return {"base": TITLES[match[0]], "similares": [TITLES.get(int(i), str(i)) for i in ids[0] if i != match[0]][:k]}


TOOLS = {"recommend_for_user": tool_recommend_for_user, "search_by_genre": tool_search_by_genre, "similar_to": tool_similar_to}
TOOL_SPECS = [
    {"name": "recommend_for_user", "description": "Recomendaciones personalizadas de CineMatch para un user_id.",
     "input_schema": {"type": "object", "properties": {"user_id": {"type": "integer"}, "k": {"type": "integer"}}, "required": ["user_id"]}},
    {"name": "search_by_genre", "description": f"Títulos populares de un género. Géneros: {', '.join(GENRES[1:])}.",
     "input_schema": {"type": "object", "properties": {"genre": {"type": "string"}, "k": {"type": "integer"}}, "required": ["genre"]}},
    {"name": "similar_to", "description": "Títulos parecidos a uno dado (búsqueda por subcadena del título).",
     "input_schema": {"type": "object", "properties": {"title": {"type": "string"}, "k": {"type": "integer"}}, "required": ["title"]}},
]

In [ ]:
import re


def agent_llm(question, user_id, max_turns=6):
    import anthropic
    client = anthropic.Anthropic()
    model = os.environ.get("CINEMATCH_LLM_MODEL", "claude-opus-5-5")
    system = (f"Eres el asistente de CineMatch. El usuario actual tiene user_id={user_id}. Usa las herramientas para "
              "basar tus respuestas en el catálogo real; no inventes títulos. Responde en español, breve.")
    messages = [{"role": "user", "content": question}]
    for _ in range(max_turns):
        resp = client.messages.create(model=model, max_tokens=4000, system=system, tools=TOOL_SPECS, messages=messages)
        if resp.stop_reason == "refusal":
            return "No puedo ayudar con esa petición."
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                try:
                    out, err = TOOLS[b.name](**b.input), False
                except Exception as e:
                    out, err = {"error": str(e)}, True
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(out, ensure_ascii=False), "is_error": err})
        messages.append({"role": "user", "content": results})
    return "(límite de turnos alcanzado)"


def agent_fallback(question, user_id):
    q = question.lower()
    if m := re.search(r"(?:como|parecid[ao]s? a)\s+(.+)", q):
        return tool_similar_to(m.group(1).strip(" ?¿."))
    for g in GENRE_IDX:
        if g in q and g != "unknown":
            return tool_search_by_genre(g)
    return tool_recommend_for_user(user_id)


USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
for q in ["¿Qué me recomiendas para esta noche?", "Quiero algo de Sci-Fi", "Algo parecido a Star Wars"]:
    try:
        ans = agent_llm(q, u0) if USE_LLM else agent_fallback(q, u0)
    except Exception as e:
        print("LLM no disponible, uso fallback:", repr(e)[:80]); ans = agent_fallback(q, u0)
    print(f"🧑 {q}\n🤖 {ans}\n")
E["bonus"] = True

In [ ]:
# E7 · Scorecard automático
proc.terminate()
score = pd.DataFrame([
    ("E1 Datos", E.get("E1", False), 10), ("E2 Retrieval", E.get("E2", False), 15), ("E3 Ranking", E.get("E3", False), 15),
    ("E4 Re-ranking", E.get("E4", False), 10), ("E5 Serving", E.get("E5", False), 15), ("E6 MLOps", E.get("E6", False), 15),
    ("E7 Informe (design doc abajo)", True, 10), ("⭐ Bonus agente", E.get("bonus", False), 10)], columns=["entregable", "cumple", "puntos"])
score["obtenidos"] = score.cumple * score.puntos
display(score)
print(f"TOTAL: {score.obtenidos.sum()} / 100 (+10 bonus)")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(score.entregable, score.puntos, color="#eaecee"); ax.barh(score.entregable, score.obtenidos, color="#58d68d")
ax.invert_yaxis(); ax.set_title("Scorecard del capstone"); plt.show()

### E7 · *Design doc* (plantilla — ≤ 1 página)

**1. Problema y métricas.** North star: … · online: … · guardrails: … · offline proxy: NDCG@10 en split temporal (justifica la correlación).

**2. Arquitectura.** Fuentes (PureSVD, two-tower logQ, co-ocurrencia, popularidad) → LambdaMART (N features PIT) → MMR (λ = …) → FastAPI (caché + fallback). Diagrama del módulo 18 §1.

**3. Resultados.** Tabla: recall por fuente, NDCG por etapa, ILD, p99, paridad. Ablación de features.

**4. Trade-offs.** Por qué estas fuentes y no otras; λ de MMR; coste de reentrenar el two-tower vs ganancia (stale vs fresh).

**5. Riesgos.** Feedback loops, cold start de estrenos, skew, sesgo de popularidad, privacidad del asistente.

**6. Siguientes pasos.** A/B con interleaving como criba (mód. 15), bandits para artwork (14), SASRec como fuente secuencial (09), semantic IDs (11), CT con Prefect (17).

## 🚀 Retos extra (nivel staff)

1. Añade **SASRec** (módulo 09) como quinta fuente y mide su contribución marginal al recall de la unión.
2. Sustituye LambdaMART por un **ranker multi-tarea** (MMoE: P(ver) y rating esperado) y fusiona objetivos con pesos elegidos por *off-policy evaluation*.
3. Simula un **A/B con interleaving** entre `capstone-v1` y una variante sin two-tower (módulo 15) usando un simulador de clics calibrado con el test.
4. Despliega con el `reference_stack/` del módulo 16 (Redis + Redpanda + Prometheus) y conecta el flow de CT del módulo 17.
5. Sirve el two-tower con **Triton** (ONNX) y compara p99 frente a TorchScript embebido.
6. Reescribe el agente con **LangGraph** (módulo 12) con memoria de conversación y un nodo de *guardrails* que verifique que todos los títulos mencionados existen en el catálogo.

## 🤔 Reflexión final

- ¿Qué parte de tu sistema aporta más NDCG por milisegundo de latencia? ¿Y por euro de cómputo?
- Si solo pudieras mantener **dos** fuentes de candidatos, ¿cuáles y por qué (usa la ablación)?
- ¿Qué métrica online usarías para decidir el λ de MMR, y cómo protegerías el experimento de efectos de novedad?
- ¿Qué cambiaría en tu diseño si CineMatch tuviera 10⁸ ítems (UGC) en lugar de 10⁴?